# PersonaPlex Launcher — RTX 5090 / A100

Goal command (runs last, after setup):

```
%cd personaplex
!python -m moshi_local.moshi.server --web-search-enabled --no-compressor-4bit --checkpoint-dir moshi_local/lora --rag-index-dir rag_index
```

Every cell above the launch section exists only to make that command work — installing exactly what `moshi_local/moshi/server.py` imports, nothing more. The PyTorch build (`cu128`) supports both Blackwell (RTX 5090) and Ampere (A100), so this notebook runs unchanged on either GPU.

In [ ]:
!nvidia-smi

In [ ]:
!apt-get update -qq && apt-get install -y -qq libopus-dev

In [ ]:
# Fresh, matched versions avoid a base-image torch build that predates Blackwell (sm_120) support.
!pip uninstall -y -q torch torchvision torchaudio transformers sentence-transformers sentencepiece accelerate safetensors

In [ ]:
!pip install torch==2.8.0 torchvision torchaudio --index-url https://download.pytorch.org/whl/cu128

In [ ]:
!pip install "numpy>=1.26,<2.2" "einops==0.7" "sphn>=0.1.4,<0.2" "aiohttp>=3.10.5,<3.11" "sentencepiece==0.2.0" "safetensors>=0.4.0,<0.5" "transformers==4.52.4" "sentence-transformers==4.1.0" accelerate huggingface_hub peft faiss-cpu

In [ ]:
# --no-deps: the moshi package on PyPI wants sphn>=0.2, which conflicts with the
# sphn<0.2 that moshi_local (this fork) requires. We already installed the sphn we need above.
!pip install --no-deps moshi

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("Compute capability:", torch.cuda.get_device_capability(0))

In [ ]:
import os
from getpass import getpass
from huggingface_hub import login

token = os.environ.get("HF_TOKEN") or getpass("Hugging Face token (needed for the gated nvidia/personaplex-7b-v1 repo): ")
os.environ["HF_TOKEN"] = token
login(token=token)

## Launch

Ensures `rag_index` exists next to `personaplex` (copied from the repo-root `rag_index` if needed), then runs the target command.

In [ ]:
%cd personaplex

In [ ]:
import os, shutil

if not os.path.isdir("rag_index") and os.path.isdir("../rag_index"):
    shutil.copytree("../rag_index", "rag_index")

assert os.path.isdir("rag_index"), "rag_index not found"
assert os.path.isdir("moshi_local/lora"), "LoRA checkpoint not found at moshi_local/lora"
print("rag_index and checkpoint ready")

In [ ]:
!python -m moshi_local.moshi.server --web-search-enabled --no-compressor-4bit --checkpoint-dir moshi_local/lora --rag-index-dir rag_index